<a href="https://colab.research.google.com/github/Rosary-mom/7gaben-hugel/blob/main/02_AlphaFold_Human_Proteins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 02 · AlphaFold DB: menschliche Proteine

Lädt ein Modell der [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/) (EMBL-EBI) über die UniProt-ID, ermittelt die aktuelle Dateiversion über die Prediction-API und färbt die Struktur in py3Dmol nach **pLDDT** (B-Faktor der PDB-Datei).

Struktur-Vorhersage; mögliche Wirkungen sind Gegenstand unserer F&E-Hypothesen (H1/H²).


## Quellen und Lizenz

- Modelle und API: [AlphaFold DB / EMBL-EBI](https://alphafold.ebi.ac.uk/), Lizenz [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- Methode: Jumper, J. et al. Highly accurate protein structure prediction with AlphaFold. *Nature* 596, 583–589 (2021). [doi:10.1038/s41586-021-03819-2](https://doi.org/10.1038/s41586-021-03819-2)
- Dateimuster (Version kommt aus der API, nicht fest v4): `https://alphafold.ebi.ac.uk/files/AF-<UniProt>-F1-model_vN.pdb`
- API: `https://alphafold.ebi.ac.uk/api/prediction/<UniProt>`
- Einträge: [AF-P00367](https://alphafold.ebi.ac.uk/entry/P00367), [AF-P02452](https://alphafold.ebi.ac.uk/entry/P02452), [AF-P04406](https://alphafold.ebi.ac.uk/entry/P04406)

pLDDT-Farben (wie in der AlphaFold DB): sehr hoch (>90) blau, hoch (70–90) cyan, niedrig (50–70) gelb, sehr niedrig (<50) orange.


In [ ]:
# In Colab beim ersten Lauf:
# %pip install -q py3Dmol

import json
import urllib.request
import py3Dmol

# Geprüfte menschliche UniProt-IDs (Stand Abruf 18.11.2026, Europe/Berlin):
# P00367  GLUD1   Homo sapiens  Glutamate dehydrogenase 1, mitochondrial
# P02452  COL1A1  Homo sapiens  Collagen alpha-1(I) chain
# P04406  GAPDH   Homo sapiens  Glyceraldehyde-3-phosphate dehydrogenase
EXAMPLES = [
    ("P00367", "GLUD1", "Glutamat-Dehydrogenase 1, mitochondrial"),
    ("P02452", "COL1A1", "Kollagen alpha-1(I)-Kette"),
    ("P04406", "GAPDH", "Glycerinaldehyd-3-phosphat-Dehydrogenase"),
]


def alphafold_model(uniprot_id: str) -> dict:
    """Aktuelle Modell-URL über die AlphaFold-API, Fallback auf das Dateimuster."""
    uid = uniprot_id.strip().upper()
    api = f"https://alphafold.ebi.ac.uk/api/prediction/{uid}"
    req = urllib.request.Request(
        api, headers={"Accept": "application/json", "User-Agent": "rosary-mediteam-notebook"}
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        recs = json.load(resp)
    rec = recs[0] if isinstance(recs, list) else recs
    version = rec.get("latestVersion") or rec.get("modelVersion") or 4
    pdb_url = rec.get("pdbUrl") or (
        f"https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v{version}.pdb"
    )
    return {
        "uniprot": uid,
        "version": version,
        "pdb_url": pdb_url,
        "entry": rec.get("entryId") or f"AF-{uid}",
        "gene": rec.get("gene") or rec.get("uniprotGene") or "",
        "name": rec.get("uniprotDescription") or "",
        "organism": rec.get("organismScientificName") or rec.get("organism") or "",
    }


def _colorfunc_js() -> str:
    return (
        "function(atom){"
        "if(atom.b>90) return '#0053D6';"
        "if(atom.b>70) return '#65CBF3';"
        "if(atom.b>50) return '#FFDB13';"
        "return '#FF7D45';}"
    )


def view_human_alphafold(uniprot_id: str, width=640, height=480):
    meta = alphafold_model(uniprot_id)
    req = urllib.request.Request(
        meta["pdb_url"], headers={"User-Agent": "rosary-mediteam-notebook"}
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        pdb = resp.read().decode("utf-8", errors="replace")
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb, "pdb")
    view.setStyle({"cartoon": {"colorfunc": _colorfunc_js()}})
    view.zoomTo()
    print(
        f"{meta['uniprot']}  {meta['gene']}  {meta['organism']}\n"
        f"{meta['name']}\n"
        f"AlphaFold-DB-Version {meta['version']}  {meta['pdb_url']}\n"
        f"Eintrag https://alphafold.ebi.ac.uk/entry/{meta['uniprot']}\n"
        f"Faerbung: pLDDT (B-Faktor). Struktur-Vorhersage (AlphaFold DB)."
    )
    view.show()
    return meta


# Ein Modell anzeigen (ID aendern oder die Beispielliste durchlaufen):
meta = view_human_alphafold("P00367")


Weitere geprüfte Beispiele (gleiche Funktion, nur die ID tauschen oder die folgende Zelle ausführen):

- `P02452` · COL1A1 · Kollagen I alpha1
- `P04406` · GAPDH


In [ ]:
# Optional: die uebrigen Beispiele nacheinander anzeigen
for uid, gene, name in EXAMPLES[1:]:
    print("----")
    view_human_alphafold(uid)


## * Grenzen

Ein `*`-Fenster klassifiziert eine Einschränkung. Es sperrt den Lauf nicht.

| Fenster | Klasse | Text |
|---|---|---|
| `*modell` | Datenbank | Die Struktur kommt aus der AlphaFold DB (EMBL-EBI). Sie ist eine Vorhersage, kein Kristall- oder NMR-Assay und kein selbst gerechneter AlphaFold-Lauf. |
| `*version` | API | Die Dateiversion (am 18.11.2026: v6, nicht fest v4) wird über `https://alphafold.ebi.ac.uk/api/prediction/<ID>` ermittelt. |
| `*plddt` | Skala | Die Färbung folgt dem pLDDT in der B-Spalte. Das ist eine Konfidenz der Vorhersage, kein Assay-Messwert einer Anwendung. |
| `*h1` | Umfang | Mögliche Wirkungen sind Gegenstand unserer F&E-Hypothesen (H1/H²), nicht Ergebnis dieser Zelle. |
